# SocialAuto analytics scratchpad

Ad-hoc queries against `social-postgres`. The `DATABASE_URL` env var is
injected by docker-compose (psycopg2, sync driver).

**First run**: execute the install cell below once, then restart the kernel.

In [1]:
%pip install -q sqlalchemy psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(os.environ["DATABASE_URL"])

def q(sql: str, **params) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

## Recent published posts

In [3]:
q("""
SELECT substring(p.id::text, 1, 8) AS pid, sa.platform, pt.status,
       pt.platform_url, pt.published_at, left(p.content_text, 60) AS preview
FROM post_targets pt
JOIN posts p ON p.id = pt.post_id
JOIN social_accounts sa ON sa.id = pt.social_account_id
WHERE pt.published_at > now() - interval '7 days'
ORDER BY pt.published_at DESC
""")

,pid,platform,status,platform_url,published_at,preview
0,8153e7da,linkedin,published,https://www.linkedin.com/feed/update/urn:li:sh...,2026-09-24 23:15:59.971046+00:00,Your shop's numbers shouldn't be a mystery. cl...
1,52dce3ab,linkedin,published,https://www.linkedin.com/feed/update/urn:li:sh...,2026-09-24 23:01:00.874819+00:00,Small teams shouldn't spend their week on serv...
2,88ab4fa6,instagram,published,https://www.instagram.com/p/18490105867098073,2026-09-24 21:02:33.701531+00:00,Hosting without servers for small teams. We he...
3,fb3a4a0f,facebook,published,https://www.facebook.com/1163886186808102_1221...,2026-09-24 15:33:28.305270+00:00,"Your website should win customers, not create ..."
4,fb3a4a0f,instagram,published,https://www.instagram.com/p/18139070968719985,2026-09-24 15:32:58.349325+00:00,"Your website should win customers, not create ..."
5,347da4d0,linkedin,published,https://www.linkedin.com/feed/update/urn:li:ug...,2026-09-24 15:28:17.163875+00:00,"Your website should win customers, not create ..."
6,6aa11c99,linkedin,published,https://www.linkedin.com/feed/update/urn:li:sh...,2026-09-24 13:12:59.779513+00:00,Skip server setup and build faster with Cloudl...
7,ed722f64,threads,published,https://www.threads.net/@cloudless.gr/post/178...,2026-09-24 09:01:31.394396+00:00,Shipping to production shouldn't take a weeken...
8,75a95bda,instagram,published,https://www.instagram.com/p/18067155596770144,2026-09-23 23:00:54.734171+00:00,Cloud hosting without the homework.\nWe run th...
9,a9d85fb3,threads,published,https://www.threads.net/@cloudless.gr/post/180...,2026-09-23 15:00:59.472621+00:00,Save 12 hours a week on server setup.\nCloudle...


## Latest follower counts per account

In [4]:
q("""
SELECT DISTINCT ON (sa.id) sa.platform, sa.username, sa.account_type,
       fs.followers, fs.captured_at
FROM follower_snapshots fs
JOIN social_accounts sa ON sa.id = fs.social_account_id
ORDER BY sa.id, fs.captured_at DESC
""")

,platform,username,account_type,followers,captured_at
0,threads,cloudless.gr,person,1,2026-09-24 23:13:46.299969+00:00
1,linkedin,baltzakis.themis@gmail.com,person,15,2026-09-24 14:38:01.670178+00:00
2,instagram,cloudless.gr,business,6,2026-09-24 23:13:24.105036+00:00
3,facebook,Themistoklis Baltzakis,user,0,2026-09-21 03:29:48.400462+00:00
4,linkedin,cloudless-gr,organization,1,2026-09-24 23:14:03.483168+00:00
5,twitter,TBaltzakis,person,0,2026-09-23 23:12:12.055359+00:00
6,facebook,cloudless.gr,page,1,2026-09-24 23:14:21.401314+00:00
7,tiktok,cloudless.gr,person,3,2026-09-22 21:13:50.639258+00:00
8,facebook,Cloudless.gr,page,0,2026-09-23 23:53:12.752046+00:00


## Engagement by publish hour (Athens) — the timing buckets behind the brief

In [5]:
q("""
SELECT sa.platform,
       extract(hour FROM p.published_at AT TIME ZONE 'Europe/Athens')::int AS hr,
       count(*) AS posts,
       round(avg(pas.engagement_rate)::numeric, 2) AS avg_er
FROM post_analytics_snapshots pas
JOIN posts p ON p.id = pas.post_id
JOIN post_targets pt ON pt.post_id = p.id AND pt.social_account_id = pas.social_account_id
JOIN social_accounts sa ON sa.id = pas.social_account_id
WHERE pas.captured_at > now() - interval '30 days'
GROUP BY 1, 2 HAVING count(*) >= 2
ORDER BY 1, 4 DESC
""")

,platform,hr,posts,avg_er
0,facebook,10,173,0.06
1,facebook,17,459,0.00
2,facebook,4,532,0.00
3,facebook,12,379,0.00
4,facebook,22,448,0.00
5,facebook,18,384,0.00
6,facebook,23,447,0.00
7,facebook,2,399,0.00
8,facebook,16,373,0.00
9,facebook,21,408,0.00
